create spark session
 

In [0]:
# Create a Spark session
# SparkSession is the entry point to Spark functionality
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName('my spark')  # Set a name for the application
    .getOrCreate()         # Get an existing session or create a new one
    
)


In [0]:
spark

In [0]:
# Define sample employee data as a list of lists.
# Each inner list represents one employee record with the following columns:
#   [employee_id, department_id, name, age, gender, salary, hire_date]
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

# Define the schema for the employee DataFrame using a DDL-formatted string.
# All columns are stored as strings (age, salary, etc. can be cast later as needed).
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"


In [0]:
# Create a DataFrame from the sample employee data and schema defined earlier.
# spark.createDataFrame takes the raw data and applies the DDL-formatted schema string.
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

# Display the DataFrame contents in a rich interactive table format in the notebook output.
display(emp)

In [0]:
#show data(action)
emp.show()

In [0]:
emp_final=emp.where("salary>50000")
emp_final.show()

In [0]:
# for schema 
emp.printSchema()
emp.schema


In [0]:
# Import PySpark data types needed to define a schema programmatically.
#   StructType  – represents a row schema (a collection of fields)
#   StructField – represents a single column inside the StructType
#   IntegerType – maps to the Spark 'int' data type
#   StringType  – maps to the Spark 'string' data type
from pyspark.sql.types import StructType,StructField,IntegerType,StringType

# A schema can be defined using a DDL-formatted string (similar to SQL column definitions).
# This is a quick, readable way to specify column names and their data types.
schema_string='name string, age int'

# Alternatively, a schema can be built programmatically using StructType and StructField.
# Each StructField takes: column name, data type, and whether the column is nullable (True = allows nulls).
schema_park=StructType([
    StructField("name",StringType(),True),   # 'name' column of type string, nullable
    StructField("age",IntegerType(),True),   # 'age' column of type int, nullable
])

In [0]:

# Columns and expression
from pyspark.sql.functions import col, expr

emp["salary"]   

In [0]:
# print the employee_id, name, age, and salary columns from the emp DataFrame
# Import column-expression helpers: col() selects a column by name, expr() parses a SQL expression string
from pyspark.sql.functions import col, expr
# emp.show()  # (optional) show the full emp DataFrame for reference
# Select only the needed columns using three different approaches:
#   col('employee_id')  – select via the col() function
#   expr('name')        – select via a SQL expression string
#   emp.salary / emp.age – select via DataFrame attribute (Column object)
emp_filter = emp.select(col('employee_id'), expr('name'), emp.salary, emp.age)
# Trigger the action and display the selected columns
emp_filter.show()

In [0]:
# Rename employee_id as emp_id and cast age from string to int.
# Using expr() for column aliasing and type casting, and DataFrame attribute for name and salary.
emp_casted=emp.select(expr("employee_id as emp_id"),emp.name,expr("cast(age as int) as age"),emp.salary)
emp_casted.show()
#

In [0]:
# filter age > 30
emp_final=emp_casted.select(emp.age,emp.emp_id,emp.name,emp.salary).where(emp.age>30)
emp_final.show()

In [0]:
# Show the original employee DataFrame (all columns, all rows)
emp.show()

# Show the filtered DataFrame with selected columns: employee_id, name, salary, age
emp_filter.show()

# Show the final DataFrame after filtering employees with age > 30
emp_final.show()

# Show the casted DataFrame with emp_id alias and age cast to int
emp_casted.show()

In [0]:
emp_age_double=emp.selectExpr("employee_id","name","cast(salary as double) as salary","age")
emp_age_double.show()
 # or 
from pyspark.sql.functions import col,cast
emp_casted_age=emp.select('employee_id','name','age',col("salary").cast("double"))
emp_casted_age.printSchema()

In [0]:
emp_casted_age.show()


In [0]:
# add tax col as salry*0.2
emp_tax=emp_casted_age.withColumn('tax',col('salary')*0.2)
emp_tax.show()

In [0]:
# Add static (literal) value columns to the emp_tax DataFrame.
# lit() creates a Column with a constant value — useful for adding fixed/default columns.
from pyspark.sql.functions import lit

# withColumn() adds a new column (or replaces an existing one of the same name).
# Here we add two new columns with constant values:
#   'columnOne' – integer literal 500 (applied to every row)
#   'columnTwo' – string literal 'two' (applied to every row)
# Multiple withColumn calls can be chained; each extends the DataFrame schema.
emp_new_col = emp_tax.withColumn('columnOne', lit(500)).withColumn('columnTwo', lit('two'))

# Trigger the action to display the updated DataFrame with the new columns
emp_new_col.show()


In [0]:
# Rename an existing column using withColumnRenamed().
# withColumnRenamed() takes two arguments:
#   - the current column name (employee_id)
#   - the new column name (emp_id)
# It returns a new DataFrame with the specified column renamed; the original DataFrame is unchanged.

emp_1 = emp_new_col.withColumnRenamed('employee_id', 'emp_id')

# Trigger the action to display the DataFrame with the renamed column
emp_1.show()


In [0]:
# ---------------------------------------------------------------------------
# Renaming a column to a name that contains a space.
#
# withColumnRenamed('columnTwo', 'column two') changes the column name from
# 'columnTwo' to 'column two' (note the space in the new name).
#
# IMPORTANT: Column names with spaces are generally discouraged because they
# can cause issues downstream. For example, if a downstream consumer expects
# a column named 'columnTwo' (without a space), renaming it to 'column two'
# will break references that use dot-notation or SQL-style column access.
# Always verify that downstream data consumers can handle space-containing
# column names before applying such a rename.
# ---------------------------------------------------------------------------

# Rename 'columnTwo' to 'column two' (with a space) in the emp_1 DataFrame.
emp_2 = emp_1.withColumnRenamed('columnTwo', 'column two')

# Trigger the action to display the DataFrame with the renamed column.
emp_2.show()

In [0]:
# remove colums 
emp_dropped=emp_new_col.drop('columnTwo')
emp_dropped.show()


In [0]:
# filter data 
emp_filtered=emp_dropped.where('tax>10000')
emp_filtered.show()




In [0]:
# limit 
emp_limited=emp_filtered.limit(5)
emp_limited.show()

In [0]:
# add n number of columns at once using withColumns()
# withColumns() takes a dict of {column_name: expression}
from pyspark.sql.functions import col, lit

columns = {
    "tax": col('salary') * 0.2,
    "columnOne": lit(500),
    "columnTwo": lit('two')
}

emp_finaldf = emp.withColumns(columns)
emp_finaldf.show()